# Session 2: Python with chemical data

This notebook works through four pieces of Python on three chemical datasets: the UV-Vis calibration series from session 1, a mass spectrum peak list, and a table of rate constants. In each section you **read** a cell, **run** it, **modify** it, and **explain** what it printed. Today Claude Code is for explaining code and error messages, and the prompts to type are written into the notebook.

Three cells are marked **[TASK]**, and the notebook does not run to the end until you have completed all three. The first is an error on purpose.

**To submit.** Put your names in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 2 assignment on Canvas by 1:30 pm on Thursday, October 8. A pair submits one notebook with both names. It is scored 3 or 0, and it earns the 3 points if it runs from top to bottom after a restart.

*Names (both, if you work as a pair):*



## Setup and imports

All libraries the notebook uses are imported in the next cell: the standard library first, then NumPy, Matplotlib, and the one SciPy function the notebook needs. The cell prints nothing when it works.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

## Parameters

Every number that encodes a choice or a physical constant is set here, once, with its unit and its reason in a comment, so a later cell never hides one. You change `THRESHOLD` in section 2. The cell prints nothing.

In [ ]:
LINEAR_LIMIT       = 1.20     # AU; top of the spectrophotometer's linear range, from session 1
THRESHOLD          = 0.05     # fraction of the base peak; smallest peak kept in section 2
PREFACTOR          = 2.0e11   # 1/s; Arrhenius prefactor used to simulate the rate table
ACTIVATION_ENERGY  = 80.0e3   # J/mol; activation energy used to simulate the rate table
GAS_CONSTANT       = 8.314    # J/(mol K)
UNKNOWN_ABSORBANCE = 0.742    # AU; an unknown sample measured on the same spectrophotometer

## 1. Values, types and lists

Every value in Python has a type, and the type decides what you can do with it. The next cell types the six standards from session 1 as two lists, i.e., ordered sequences of values in square brackets, and prints the type of four things: a list, one item of that list, a count, and a unit. The output is four lines: `list`, `float`, `int` and `str`.

In [ ]:
concentrations = [0.0, 5.0, 10.0, 15.0, 20.0, 25.0]      # µM
absorbances    = [0.002, 0.296, 0.599, 0.897, 1.192, 1.405]  # AU
n_standards    = len(absorbances)
unit           = "AU"

print(type(absorbances))
print(type(absorbances[0]))
print(type(n_standards))
print(type(unit))

Python counts positions in a list from 0, so the third standard is at index 2. The next cell prints it with an f-string, i.e., a string with `f` before the opening quote whose `{}` fields are filled with values; `:.3f` inside a field means three decimal places. The output is `10.0 µM: 0.599 AU`.

In [ ]:
print(f"{concentrations[2]:.1f} µM: {absorbances[2]:.3f} {unit}")

**[TASK 1] An error on purpose.** The next cell tries to print the 25 µM standard and stops with an error. Run it, then copy the whole error message, from its first line to its last, and paste it into Claude Code below this prompt:

```text
Explain this error message from section 1 of sessions/S02-python-with-chemical-data/activity.ipynb. Say which line it points to and why it fails. Do not fix it.
```

Then fix the cell yourself so it prints `25.0 µM: 1.405 AU`. There are two ways: a positive index, or a negative index, which counts from the end of the list, so that `-1` is the last item.

In [ ]:
print(f"{concentrations[6]:.1f} µM: {absorbances[6]:.3f} {unit}")

**Explain.** In one or two sentences in the next cell, say why the last of six standards is at index 5, and which line of the error message told you what was wrong.

*Your explanation:*



## 2. Loops and list comprehensions

A loop repeats one step for every item of a list. The next cell walks through the standards with `zip`, which pairs the first concentration with the first absorbance, the second with the second, and so on, and marks any standard above `LINEAR_LIMIT`. The output is six lines, and only the last is marked.

In [ ]:
for c, a in zip(concentrations, absorbances):
    if a > LINEAR_LIMIT:
        note = "  above the linear range"
    else:
        note = ""
    print(f"{c:5.1f} µM  {a:.3f} AU{note}")

A list comprehension, i.e., a loop written inside square brackets that builds a new list, does the same selection in one line. The next cell keeps the absorbances at or below `LINEAR_LIMIT`. The output is a list of five values ending at 1.192, and a count.

In [ ]:
in_range = [a for a in absorbances if a <= LINEAR_LIMIT]
print(in_range)
print(f"{len(in_range)} of {n_standards} standards are in the linear range")

Before you write a comprehension of your own, ask Claude Code to take this one apart:

```text
Explain the line that defines in_range in section 2 of sessions/S02-python-with-chemical-data/activity.ipynb, one piece at a time: what the for, the in and the if each do. Do not change the notebook.
```

Now a mass spectrum. `data/ms_peaks.csv` is the peak list from a simulated electrospray mass spectrum of a caffeine standard: one row per peak, with its m/z and its intensity in counts. The next cell loads it and turns it into a list of (m/z, intensity) pairs, i.e., tuples, then loops over the pairs to find the base peak, the most intense peak in the spectrum, which relative intensities are measured against. The output is the number of peaks, the first pair, and the base peak, which is protonated caffeine at m/z 195.088.

In [ ]:
peak_table = np.loadtxt("data/ms_peaks.csv", delimiter=",", skiprows=1)
peaks = list(zip(peak_table[:, 0].tolist(), peak_table[:, 1].tolist()))  # (m/z, counts)

base_mz, base_intensity = peaks[0]
for mz, intensity in peaks:
    if intensity > base_intensity:
        base_mz, base_intensity = mz, intensity

print(f"{len(peaks)} peaks; the first is {peaks[0]}")
print(f"base peak: m/z {base_mz:.4f}, {base_intensity:.0f} counts")

**[TASK 2] Your first list comprehension.** Replace the `...` in the next cell with a list comprehension that keeps the peaks whose intensity is at least `THRESHOLD` times `base_intensity`. It has the shape of `in_range` above, except that each item is a pair, so it begins `[(mz, intensity) for mz, intensity in peaks if`. The loop that prints the result is written for you. With `THRESHOLD = 0.05` it prints five peaks, the first at m/z 138.0663. If you are stuck, ask Claude Code to explain the `in_range` line again rather than to write this one.

In [ ]:
major_peaks = ...  # [TASK 2] (m/z, intensity) pairs with intensity >= THRESHOLD * base_intensity

for mz, intensity in major_peaks:
    print(f"m/z {mz:9.4f}  {100 * intensity / base_intensity:5.1f}% of the base peak")
print(f"{len(major_peaks)} peaks at or above {THRESHOLD:.0%} of the base peak")

**Modify.** Change `THRESHOLD` in the Parameters cell from 0.05 to 0.01, run the Parameters cell and then the cell above, and count the peaks. Set it back to 0.05 when you are done.

**Explain.** At 5% the five peaks are caffeine ions: protonated caffeine at m/z 195.088, a peak at 196.091, the sodium adduct at 217.069, a fragment at 138.066 and the protonated dimer at 389.168. In the next cell, say how many peaks the list holds at 1%, and what the spacing of 1.003 between m/z 195.088 and 196.091 tells you about the ion at 196.091.

*Your explanation:*



## 3. Functions and docstrings

A function packages a calculation so that it is written once and used many times, and its docstring, i.e., the text in triple quotes under the `def` line, says what goes in and what comes out, with units. The next cell defines the Arrhenius equation,

$$k = A \, e^{-E_a / RT},$$

which says that the rate constant $k$ equals the prefactor $A$ times the exponential of minus the activation energy $E_a$ divided by the gas constant $R$ times the temperature $T$. The cell prints nothing.

In [ ]:
def arrhenius(temperature, prefactor, activation_energy):
    """Rate constant from the Arrhenius equation.

    Parameters
    ----------
    temperature : float or numpy.ndarray
        Absolute temperature (K).
    prefactor : float
        Pre-exponential factor (1/s).
    activation_energy : float
        Activation energy (J/mol).

    Returns
    -------
    float or numpy.ndarray
        First-order rate constant (1/s).
    """
    return prefactor * np.exp(-activation_energy / (GAS_CONSTANT * temperature))

`data/arrhenius_rates.csv` holds first-order rate constants for one reaction measured at seven temperatures from 283 to 343 K. The table is simulated from `PREFACTOR` and `ACTIVATION_ENERGY` with a 3% measurement scatter, so the function should reproduce it. The next cell loads the table and calls the function at the first, fourth and last temperatures. The output is three lines, each prediction within 3% of the measured value.

In [ ]:
rate_table    = np.loadtxt("data/arrhenius_rates.csv", delimiter=",", skiprows=1)
temperature   = rate_table[:, 0]  # K
rate_constant = rate_table[:, 1]  # 1/s

for row in [0, 3, 6]:
    predicted = arrhenius(temperature[row], PREFACTOR, ACTIVATION_ENERGY)
    difference = 100 * (predicted - rate_constant[row]) / rate_constant[row]
    print(f"{temperature[row]:.0f} K: measured {rate_constant[row]:.3e} 1/s, "
          f"predicted {predicted:.3e} 1/s ({difference:+.1f}%)")

**Modify.** Activation energies are usually quoted in kJ/mol. In the next cell, replace `ACTIVATION_ENERGY` with `80.0`, the same energy in kJ/mol, and run it.

In [ ]:
wrong_units = arrhenius(temperature[3], PREFACTOR, ACTIVATION_ENERGY)
print(f"{temperature[3]:.0f} K: {wrong_units:.3e} 1/s, measured {rate_constant[3]:.3e} 1/s")

**Explain.** Python printed no error. In the next cell, say roughly how many times larger than the measured value the new rate constant is, and which line of the docstring would have stopped you.

*Your explanation:*



**[TASK 3] Your first function.** Section 4 needs the inverse of the calibration from the session 1 notes: the concentration of a sample is its absorbance minus the blank, divided by the slope, $c = (A - A_{\mathrm{bl}})/m$. Complete the function in the next cell. Replace each `[TASK 3]` line of the docstring with the meaning and unit, as in `arrhenius`, and replace the `...` after `return` with the calculation. To see what a good docstring holds, ask:

```text
Explain what each part of the docstring of arrhenius in section 3 of sessions/S02-python-with-chemical-data/activity.ipynb is for. Do not write the docstring for concentration_from_absorbance.
```

The cell after it checks your function on the 10 µM standard, with the slope from the session 1 notes, 0.0596 AU/µM, and a blank of 0.001 AU. It should print `10.03 µM`.

In [ ]:
def concentration_from_absorbance(absorbance, slope, blank):
    """[TASK 3] Replace this line with a one line summary.

    Parameters
    ----------
    absorbance : float or numpy.ndarray
        [TASK 3] meaning and unit
    slope : float
        [TASK 3] meaning and unit
    blank : float
        [TASK 3] meaning and unit

    Returns
    -------
    float or numpy.ndarray
        [TASK 3] meaning and unit
    """
    return ...  # [TASK 3] the concentration, from absorbance, slope and blank

In [ ]:
print(f"{concentration_from_absorbance(0.599, 0.0596, 0.001):.2f} µM")

## 4. Arrays, a fit and one plot

A NumPy array does arithmetic on every element at once, so it needs no loop. Session 1 ended with a straight line through all six standards; this section plots the residuals, i.e., each measured absorbance minus the absorbance the line predicts, which is where a wrong model shows. The next cell loads the same six standards from `data/uvvis_calibration.csv` as arrays and builds a boolean mask, i.e., an array of `True` and `False` with one entry per standard, which is the array version of `in_range` in section 2. The output is the mask and a count of 5 of 6.

In [ ]:
calibration   = np.loadtxt("data/uvvis_calibration.csv", delimiter=",", skiprows=1)
concentration = calibration[:, 0]  # µM
absorbance    = calibration[:, 1]  # AU

in_linear_range = absorbance <= LINEAR_LIMIT
print(in_linear_range)
print(f"{in_linear_range.sum()} of {absorbance.size} standards are in the linear range")

The next cell fits two lines with `linregress`: one to all six standards, and one to the five in the linear range, which indexing an array with the mask selects. It then computes the residual of every standard from each line, using the fitted intercept as the blank. The output is the slope, intercept and $R^2$ of each fit, and the $R^2$ of both is above 0.99.

In [ ]:
fit_all    = linregress(concentration, absorbance)
fit_linear = linregress(concentration[in_linear_range], absorbance[in_linear_range])

residual_all    = absorbance - (fit_all.slope * concentration + fit_all.intercept)  # AU
residual_linear = absorbance - (fit_linear.slope * concentration + fit_linear.intercept)  # AU

for label, fit in [("all six", fit_all), ("five in range", fit_linear)]:
    print(f"{label:>13}: slope {fit.slope:.4f} AU/µM, intercept {fit.intercept:.4f} AU, "
          f"R² = {fit.rvalue**2:.4f}")

The next cell plots the residuals of both fits against concentration, circles for the fit to all six standards and squares for the fit to the five in the linear range, with a dashed line at zero.

In [ ]:
fig, ax = plt.subplots()
ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
ax.plot(concentration, residual_all, "o", label="fit to all six standards")
ax.plot(concentration, residual_linear, "s", label="fit to the five in the linear range")
ax.set_xlabel("Concentration (µM)")
ax.set_ylabel("Residual absorbance (AU)")
ax.set_title("Session 2: residuals of two calibration fits")
ax.legend()
fig.tight_layout()
plt.show()

The plot shows the circles rising from -0.016 AU at 0 µM to +0.032 AU at 20 µM and dropping to -0.041 AU at 25 µM, whereas the squares lie within 0.004 AU of the dashed zero line at every standard except 25 µM, at -0.0865 AU.

The next cell uses your function from TASK 3 to convert `UNKNOWN_ABSORBANCE` to a concentration with each fit. The output is two concentrations about 0.25 µM apart, and their difference in percent.

In [ ]:
unknown_all    = concentration_from_absorbance(UNKNOWN_ABSORBANCE, fit_all.slope, fit_all.intercept)
unknown_linear = concentration_from_absorbance(UNKNOWN_ABSORBANCE, fit_linear.slope, fit_linear.intercept)

print(f"fit to all six:       {unknown_all:.2f} µM")
print(f"fit to five in range: {unknown_linear:.2f} µM")
print(f"difference: {100 * (unknown_all - unknown_linear) / unknown_linear:+.1f}%")

**Explain.** The circles in the plot form a pattern and the squares, apart from the 25 µM standard, do not. In the next cell, say which concentration you would report for the unknown and what in the plot is your reason. This is the kind of reasoning the judgment cell asks for, starting in session 3.

*Your explanation:*



## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"Matplotlib {matplotlib.__version__}")
print(f"SciPy      {scipy.__version__}")